In [ ]:
import elecu.restructure_results
import elecu.extract_values
import elecu.visualize_results
from elecu.extract_values import extract_eleccion, extract_votacion
from elecu.utils import load_std_data
import pandas as pd
from elecu.restructure_results import Standarized_Results
print("Elecu module loaded successfully.")

# Generación de resultados seccionales 2023 - Prefectos

In [ ]:
def test_standarized_registro(year=2023):
    """Carga y estandariza el registro electoral (padrón) por cantones"""
    input_folder = f"../data/csv_files/seccionales/{year}"
    standarized_results = Standarized_Results(input_folder, None)
    standarized_results.change_registro()
    test_registro = standarized_results.put_standar_geo_codes_registro_canton(drop_old=True, year=year)
    if "ELECTORES MENORES A 18" not in test_registro.columns:
        test_registro["ELECTORES MENORES A 18"] = 0
    return test_registro


def test_standarized_resultados(year=2023):
    """Carga y estandariza los resultados electorales por cantones"""
    input_folder = f"../data/csv_files/seccionales/{year}"
    standarized_results = Standarized_Results(input_folder, None)
    standarized_results.change_resultados()
    test = standarized_results.put_standar_geo_codes_results_cantones(drop_old=True, year=year)
    test_year_votacion, test_year_eleccion = standarized_results.divide_resultados()
    return test_year_votacion, test_year_eleccion, test


def apply_suffix(df, sexo):
    """Agrega sufijos (_M, _F, _T) a las columnas de candidatos según el sexo"""
    columns = df.columns.tolist()
    suffix = "_M" if sexo == "S0" else "_F" if sexo == "S1" else "_T"
    columns = columns[:2] + [col + suffix for col in columns[2:]]
    df.columns = columns
    return df


# Cargar diccionario de provincias
import os
years = os.listdir("../data/csv_files/seccionales")
years = [int(year) for year in years]
years.sort()
provincias_df = load_std_data("provincias/std_provincias.csv")
provincias_dict = provincias_df[["PROVINCIA_CODIGO", "PROVINCIA_NOMBRE"]].set_index("PROVINCIA_CODIGO").to_dict()["PROVINCIA_NOMBRE"]
# Eliminar provincias especiales
del provincias_dict["EC00"]
del provincias_dict["EC25"]

print("Funciones base cargadas correctamente")

In [ ]:
def extract_prefectos_names_and_votos(year=2023, dignidad_codigo="2", vuelta="1", agrupar_por_territorio="PROVINCIA", territorio_codigo=None, sexo="AGRUPAR"):
    """
    Extrae votos de prefectos por territorio con sus nombres y organizaciones políticas.
    """
    test_registro = test_standarized_registro(year)
    test_year_votacion, test_year_eleccion, _ = test_standarized_resultados(year)

    prefectos_year = extract_eleccion(
        test_year_eleccion,
        dignidad_codigo=dignidad_codigo,
        territorio_codigo=territorio_codigo,
        agrupar_por_territorio=agrupar_por_territorio,
        sexo=sexo,
        vuelta=vuelta
    )

    # Obtener nombres de candidatos
    candidato_nombre_opciones = ["CANDIDATO_NOMBRE", "CANDIDATO_NOMBRE_RESULTADOS"]
    if "CANDIDATO_CODIGO" in prefectos_year.columns:
        candidatos_path = f"../data/csv_files/seccionales/{year}/organizaciones_politicas/candidatos_{year}.csv"
        candidatos_df = pd.read_csv(candidatos_path)

        for candidato_nombre_opcion in candidato_nombre_opciones:
            if candidato_nombre_opcion in candidatos_df.columns:
                candidato_codigo_nombre = candidatos_df[["CANDIDATO_CODIGO", candidato_nombre_opcion]]
                break

        prefectos_year = pd.merge(prefectos_year, candidato_codigo_nombre, on="CANDIDATO_CODIGO", how="inner")
        prefectos_year = prefectos_year.rename(columns={candidato_nombre_opcion: "CANDIDATO_NOMBRE"})
        prefectos_year.drop(columns=["CANDIDATO_CODIGO"], inplace=True)
    else:
        print("CANDIDATO_NOMBRE already in the dataframe")

    # Obtener nombres de organizaciones políticas
    if "OP_CODIGO" in prefectos_year.columns:
        organizaciones_path = f"../data/csv_files/seccionales/{year}/organizaciones_politicas/organizaciones_politicas_{year}.csv"
        organizaciones_df = pd.read_csv(organizaciones_path)
        organizaciones_df = organizaciones_df[["OP_CODIGO", "OP_NOMBRE"]]
        organizaciones_df["OP_CODIGO"] = organizaciones_df["OP_CODIGO"].astype(str)
        prefectos_year["OP_CODIGO"] = prefectos_year["OP_CODIGO"].astype(str)
        prefectos_year = pd.merge(prefectos_year, organizaciones_df, on="OP_CODIGO", how="inner")
        prefectos_year.drop(columns=["OP_CODIGO"], inplace=True)

    # Agregar códigos geográficos (provincia o cantón)
    if agrupar_por_territorio == "PROVINCIA":
        provincias_df = load_std_data("provincias/std_provincias.csv")
        provincias_df = provincias_df[["PROVINCIA_CODIGO", "PROVINCIA_NOMBRE"]]
        prefectos_year = pd.merge(prefectos_year, provincias_df, on="PROVINCIA_CODIGO", how="inner")
        columns_territorio = ["PROVINCIA_CODIGO", "PROVINCIA_NOMBRE"]

    if agrupar_por_territorio == "CANTON":
        cantones_df = load_std_data("cantones/std_cantones.csv")
        cantones_df = cantones_df[["CANTON_CODIGO", "CANTON_NOMBRE"]]
        cantones_df["CANTON_CODIGO"] = cantones_df["CANTON_CODIGO"].astype(str)
        prefectos_year = pd.merge(prefectos_year, cantones_df, on="CANTON_CODIGO", how="inner")
        columns_territorio = ["CANTON_CODIGO", "CANTON_NOMBRE"]

    # PIVOT TABLE: convertir a formato ancho con candidatos como columnas
    prefectos_year = prefectos_year.pivot_table(
        index=columns_territorio,
        columns="CANDIDATO_NOMBRE",
        values="VOTOS",
        aggfunc="sum"
    ).reset_index()

    # Obtener datos de votación (blancos y nulos)
    test_year_votacion["DIGNIDAD_CODIGO"] = test_year_votacion["DIGNIDAD_CODIGO"].astype(str)
    test_year_votacion_prefectos = test_year_votacion[test_year_votacion["DIGNIDAD_CODIGO"] == dignidad_codigo]

    territorio_codigo_col = columns_territorio[0]
    territorio_nombre = columns_territorio[1]

    test_year_votacion_prefectos = test_year_votacion_prefectos[[territorio_codigo_col, "SEXO", "BLANCOS", "NULOS", "VUELTA"]]
    test_year_votacion_prefectos = test_year_votacion_prefectos[test_year_votacion_prefectos["VUELTA"] == vuelta]

    columnas_agrupar_votacion = ['SEXO', territorio_codigo_col]
    if sexo is not None:
        if sexo == 'S0' or sexo == "S1":
            test_year_votacion_prefectos = test_year_votacion_prefectos[test_year_votacion_prefectos['SEXO'] == sexo]
        if sexo == "AMBOS":
            pass
        if sexo == "AGRUPAR":
            columnas_agrupar_votacion.remove('SEXO')
        test_year_votacion_prefectos = test_year_votacion_prefectos.groupby(columnas_agrupar_votacion).agg({"BLANCOS": "sum", "NULOS": "sum"}).reset_index()

    prefectos_year = pd.merge(
        prefectos_year,
        test_year_votacion_prefectos[[territorio_codigo_col, "BLANCOS", "NULOS"]],
        on=territorio_codigo_col,
        how="inner"
    )

    # Obtener datos de electores
    columnas_agrupar = ['SEXO', territorio_codigo_col]
    if sexo is not None:
        if sexo == 'S0' or sexo == "S1":
            test_registro = test_registro[test_registro['SEXO'] == sexo]
        if sexo == "AMBOS":
            pass
        if sexo == "AGRUPAR":
            columnas_agrupar.remove('SEXO')
        test_registro_year = test_registro.groupby(columnas_agrupar)['TOTAL ELECTORES'].sum().reset_index()

    test_registro_year["TOTAL ELECTORES"] = test_registro_year["TOTAL ELECTORES"].astype(int)
    prefectos_year = pd.merge(
        prefectos_year,
        test_registro_year[[territorio_codigo_col, "TOTAL ELECTORES"]],
        on=territorio_codigo_col,
        how="inner"
    )

    prefectos_year.rename(columns={"TOTAL ELECTORES": "ELECTORES"}, inplace=True)

    # Calcular votos válidos
    prefectos_year["VOTOS VALIDOS"] = prefectos_year.iloc[:, 2:-3].sum(axis=1)

    # Reorganizar columnas: poner VOTOS VALIDOS antes de BLANCOS y NULOS
    columns = prefectos_year.columns.tolist()
    columns = columns[:2] + columns[2:-4] + ["VOTOS VALIDOS"] + columns[-4:-1]
    prefectos_year = prefectos_year[columns]

    # Calcular sufragantes
    prefectos_year["SUFRAGANTES"] = prefectos_year["VOTOS VALIDOS"] + prefectos_year["BLANCOS"] + prefectos_year["NULOS"]

    prefectos_year.drop_duplicates(subset=[territorio_codigo_col], keep="first", inplace=True)

    return prefectos_year

In [ ]:
def merge_prefectos_votes_by_sex(year, agrupar_por_territorio, territorio_codigo, dignidad_codigo="2"):
    """
    Extrae votos de prefectos para diferentes sexos y los fusiona por cantón.
    """
    combined_dfs = []
    vuelta = "1"  # Prefectos solo tienen una vuelta

    print(f"Processing year: {year}, dignidad: {dignidad_codigo}")

    # Extraer datos para cada sexo
    print(f"Extracting data for male")
    df_masc = extract_prefectos_names_and_votos(
        year=year,
        dignidad_codigo=dignidad_codigo,
        vuelta=vuelta,
        agrupar_por_territorio=agrupar_por_territorio,
        territorio_codigo=territorio_codigo,
        sexo="S0"
    )

    print(f"Extracting data for female")
    df_fem = extract_prefectos_names_and_votos(
        year=year,
        dignidad_codigo=dignidad_codigo,
        vuelta=vuelta,
        agrupar_por_territorio=agrupar_por_territorio,
        territorio_codigo=territorio_codigo,
        sexo="S1"
    )

    print(f"Extracting data for total")
    df_total = extract_prefectos_names_and_votos(
        year=year,
        dignidad_codigo=dignidad_codigo,
        vuelta=vuelta,
        agrupar_por_territorio=agrupar_por_territorio,
        territorio_codigo=territorio_codigo,
        sexo="AGRUPAR"
    )

    # Aplicar sufijos a cada dataframe
    df_masc = apply_suffix(df_masc, "S0")
    df_fem = apply_suffix(df_fem, "S1")
    df_total = apply_suffix(df_total, "AGRUPAR")

    # Fusionar los dataframes
    if agrupar_por_territorio == "PROVINCIA":
        merge_keys = ['PROVINCIA_CODIGO', 'PROVINCIA_NOMBRE']
    else:  # CANTON
        merge_keys = ['CANTON_CODIGO', 'CANTON_NOMBRE']

    df_merged = df_masc.merge(df_fem, on=merge_keys, how='outer') 
    df_merged = df_merged.merge(df_total, on=merge_keys, how='outer')

    # Agregar columnas de metadatos
    df_merged.insert(0, 'ANIO', year)
    df_merged.insert(1, 'DIGNIDAD_CODIGO', dignidad_codigo)

    return df_merged

## Generar prefectos_votes_by_canton_2023 (formato ancho por cantón y sexo)

In [ ]:
# Procesar prefectos para todas las provincias (cantonal, por sexo)
years = [2023]
df_prefectos_por_provincia = {}  # Diccionario para almacenar un DataFrame por provincia

for year in years:
    for codigo, provincia in provincias_dict.items():
        print(f"Year: {year}, Provincia: {provincia}")
        try:
            df_provincia = merge_prefectos_votes_by_sex(
                year=year,
                agrupar_por_territorio="CANTON",
                territorio_codigo=codigo,
                dignidad_codigo="2"
            )
            df_prefectos_por_provincia[codigo] = df_provincia
            print(f"✅ {provincia}: {len(df_provincia)} cantones")
        except Exception as e:
            print(f"❌ Error processing {provincia}: {str(e)}")
            df_prefectos_por_provincia[codigo] = None
            continue

# Crear DataFrame combinado cantonal
df_prefectos_total_cantonal = pd.concat(
    [df for df in df_prefectos_por_provincia.values() if df is not None],
    ignore_index=True
)

output_cantonal = os.path.abspath("../tests/prefectos_votes_by_canton_2023.csv")
df_prefectos_total_cantonal.to_csv(output_cantonal, index=False)
print(f"Archivo generado: {output_cantonal}")

## Generar resultados de prefectos 2023 en formato largo por provincia

In [ ]:
# Formato largo por provincia (ANIO, VUELTA, PROVINCIA_CODIGO, PROVINCIA_NOMBRE, CANDIDATO, VOTOS)

year = 2023
vuelta = "1"

registros = []

for codigo, provincia in provincias_dict.items():
    print(f"Procesando provincia: {codigo} - {provincia}")
    # Usamos agregación a nivel de PROVINCIA y sexo=AGRUPAR para obtener totales por candidato
    df_prov = extract_prefectos_names_and_votos(
        year=year,
        dignidad_codigo="2",
        vuelta=vuelta,
        agrupar_por_territorio="PROVINCIA",
        territorio_codigo=codigo,
        sexo="AGRUPAR",
    )

    # df_prov tiene columnas: PROVINCIA_CODIGO, PROVINCIA_NOMBRE, <candidatos>, BLANCOS, NULOS, ELECTORES, VOTOS VALIDOS, SUFRAGANTES
    id_cols = ['PROVINCIA_CODIGO', 'PROVINCIA_NOMBRE']
    value_cols = [c for c in df_prov.columns if c not in id_cols + ['BLANCOS', 'NULOS', 'ELECTORES', 'VOTOS VALIDOS', 'SUFRAGANTES']]
    # value_cols incluye columnas de candidatos
    df_long = df_prov.melt(id_vars=id_cols, value_vars=value_cols, var_name='CANDIDATO', value_name='VOTOS')

    # Añadir metadatos de año y vuelta
    df_long['ANIO'] = year
    df_long['VUELTA'] = int(vuelta)
    registros.append(df_long)

# Concatenar todas las provincias
df_prefectos_long_2023 = pd.concat(registros, ignore_index=True)

# Ordenar columnas como se solicitó
df_prefectos_long_2023 = df_prefectos_long_2023[['ANIO', 'VUELTA', 'PROVINCIA_CODIGO', 'PROVINCIA_NOMBRE', 'CANDIDATO', 'VOTOS']]

# Guardar archivo largo global en tests
output_long_global = os.path.abspath("../tests/prefectos_prefectos_2023_long.csv")
df_prefectos_long_2023.to_csv(output_long_global, index=False)
print(f"Archivo largo global generado: {output_long_global}")

# Guardar un archivo por provincia en tests/seccionales_2023 con el formato solicitado
output_dir_prov = os.path.abspath("../tests/seccionales_2023")
os.makedirs(output_dir_prov, exist_ok=True)

for codigo, provincia in provincias_dict.items():
    df_prov_long = df_prefectos_long_2023[df_prefectos_long_2023['PROVINCIA_CODIGO'] == codigo].copy()
    nombre_norm = provincia.lower().replace(' ', '_')
    filename = f"prefectos_{codigo}_{nombre_norm}_2023.csv"
    path_csv = os.path.join(output_dir_prov, filename)
    df_prov_long.to_csv(path_csv, index=False)
    print(f"✅ Archivo generado para {provincia}: {path_csv}")